# Part 2 – Air Quality: 02 Clean & Feature Engineering

Reads the raw JSON extracts produced by `01_ingest.ipynb`, applies four QA rules,  
and creates a **curated daily dataset** with one row per `(city, date)`.

Output: `data/curated/part2_airquality_curated.parquet`

### QA Rules Applied
| # | Rule |
|---|------|
| 1 | Remove exact-duplicate PM2.5 records (same city, same timestamp, same value) |
| 2 | Convert timestamps to a consistent timezone before daily aggregation |
| 3 | If a day has fewer than 12 PM2.5 measurements, set `pm25_mean` / `pm25_p90` to NaN |
| 4 | Drop days with missing PM2.5; report dropped-day count per city |

In [ ]:
import json
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings('ignore')

In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
ROOT    = Path(os.getcwd()).parent if Path(os.getcwd()).name == 'part2_airquality' else Path(os.getcwd())
RAW_AQ  = ROOT / 'data' / 'raw' / 'openaq'
RAW_WX  = ROOT / 'data' / 'raw' / 'openmeteo'
CURATED = ROOT / 'data' / 'curated'
CURATED.mkdir(parents=True, exist_ok=True)

CITIES = {
    'nyc':     {'label': 'New York City', 'tz': 'America/New_York'},
    'la':      {'label': 'Los Angeles',   'tz': 'America/Los_Angeles'},
    'chicago': {'label': 'Chicago',       'tz': 'America/Chicago'},
}

## Step 1 – Load Raw OpenAQ Data into a Flat Measurement Table

Each OpenAQ record represents one sensor's **daily aggregate** and carries:
- `value` / `summary.avg` — daily mean PM2.5 (µg/m³)
- `summary.q75`, `summary.q98` — intra-day percentile statistics
- `coverage.observedCount` — number of hourly readings aggregated into that day  
- `period.datetimeFrom.utc` — UTC timestamp of the period start
- `period.datetimeFrom.local` — local-time timestamp (city timezone)

In [ ]:
def load_flat(path: Path, city_key: str) -> pd.DataFrame:
    """Expand a single OpenAQ JSON file into one row per daily measurement record."""
    with open(path) as f:
        raw = json.load(f)

    rows = []
    for r in raw['results']:
        rows.append({
            'city_key':        city_key,
            'city':            CITIES[city_key]['label'],
            # UTC timestamp — used for deduplication and timezone conversion
            'ts_utc':          r['period']['datetimeFrom']['utc'],
            # Local timestamp string (carries offset, e.g. -05:00)
            'ts_local':        r['period']['datetimeFrom']['local'],
            'value':           r['summary']['avg'],          # daily mean PM2.5
            'q75':             r['summary']['q75'],
            'q98':             r['summary']['q98'],
            'observed_count':  r['coverage']['observedCount'],  # hourly readings in day
        })
    return pd.DataFrame(rows)


raw_frames = [load_flat(RAW_AQ / f'pm25_{k}_2025Q1.json', k) for k in CITIES]
raw_aq = pd.concat(raw_frames, ignore_index=True)

print(f'Total records loaded: {len(raw_aq)}')
raw_aq.head(3)

---
## QA Rule 1 – Remove Exact Duplicates

A duplicate is defined as a record with the **same city, UTC timestamp, and PM2.5 value**.

In [ ]:
before = len(raw_aq)
raw_aq = raw_aq.drop_duplicates(subset=['city_key', 'ts_utc', 'value'])
after  = len(raw_aq)
removed_dupes = before - after

print(f'QA Rule 1 – Duplicate removal')
print(f'  Records before : {before}')
print(f'  Duplicates removed : {removed_dupes}')
print(f'  Records after  : {after}')

---
## QA Rule 2 – Consistent Timezone for Daily Aggregation

**Timezone chosen: UTC**

Rationale: All three cities span three different UTC offsets  
(UTC−5 NYC, UTC−6 Chicago, UTC−8 Los Angeles).  
Converting every timestamp to **UTC before extracting the calendar date** ensures  
that no observation is double-counted or silently re-assigned across midnight boundaries  
due to daylight-saving transitions or mixed offsets in the raw data.

Each OpenAQ record already supplies `period.datetimeFrom.utc`, so we parse that  
field and derive the UTC calendar date directly.

In [ ]:
raw_aq['ts_utc_parsed'] = pd.to_datetime(raw_aq['ts_utc'], utc=True)
raw_aq['date_utc']      = raw_aq['ts_utc_parsed'].dt.normalize()   # midnight UTC

print('QA Rule 2 – Timezone conversion')
print('  All timestamps converted to UTC before date extraction.')
print(f'  Timezone used for daily aggregation: UTC')
print(f'  Sample UTC dates:')
print(raw_aq[['city', 'ts_local', 'ts_utc', 'date_utc']].head(4).to_string(index=False))

---
## QA Rule 3 – Minimum Coverage Threshold (< 12 Measurements → NaN)

`coverage.observedCount` records how many hourly PM2.5 readings were used to build  
each daily summary. Days with fewer than **12 observations** (< 50 % of a 24-hour day)  
are considered insufficiently sampled; their `pm25_mean` and `pm25_p90` are set to NaN.

In [ ]:
MIN_OBS = 12
low_coverage_mask = raw_aq['observed_count'] < MIN_OBS

print(f'QA Rule 3 – Low-coverage days (observedCount < {MIN_OBS})')
print(f'  Total low-coverage records: {low_coverage_mask.sum()}')
print()
per_city = raw_aq.groupby('city')['observed_count'].apply(lambda s: (s < MIN_OBS).sum())
for city, n in per_city.items():
    print(f'  {city:20s}: {n} low-coverage day(s)')

# Set pm25 fields to NaN for low-coverage days
raw_aq.loc[low_coverage_mask, 'value'] = np.nan
raw_aq.loc[low_coverage_mask, 'q75']   = np.nan
raw_aq.loc[low_coverage_mask, 'q98']   = np.nan

---
## Aggregate to Daily PM2.5 Metrics

Each record is already one daily period, so aggregation here simply organises  
the deduplicated data into a `(city, date_utc)` table and computes:
- `pm25_mean` = `summary.avg` (directly from the API daily summary)
- `pm25_p90`  = linear interpolation between q75 and q98  
  &nbsp;&nbsp;&nbsp; `p90 = q75 + (90−75)/(98−75) × (q98−q75)`  
  *(OpenAQ v3 daily endpoint does not expose q90 directly)*

In [ ]:
def safe_p90(row):
    if pd.isna(row['q75']) or pd.isna(row['q98']):
        return np.nan
    return row['q75'] + (90 - 75) / (98 - 75) * (row['q98'] - row['q75'])

raw_aq['pm25_mean'] = raw_aq['value'].round(4)
raw_aq['pm25_p90']  = raw_aq.apply(safe_p90, axis=1).round(4)

aq_daily = (
    raw_aq[['city', 'date_utc', 'pm25_mean', 'pm25_p90', 'observed_count']]
    .rename(columns={'date_utc': 'date'})
    .sort_values(['city', 'date'])
    .reset_index(drop=True)
)

print(f'AQ daily table shape: {aq_daily.shape}')
aq_daily.head(4)

---
## QA Rule 4 – Drop Days with Missing PM2.5 (No Interpolation)

Days flagged as NaN in Rule 3 are **dropped** (not filled/interpolated).  
The count of dropped days is reported per city.

In [ ]:
missing_mask = aq_daily['pm25_mean'].isna()

print('QA Rule 4 – Days dropped due to missing PM2.5 (no interpolation)')
dropped_per_city = aq_daily[missing_mask].groupby('city').size()

for city in [c['label'] for c in CITIES.values()]:
    n = dropped_per_city.get(city, 0)
    print(f'  {city:20s}: {n} day(s) dropped')

print(f'\n  Total days dropped: {missing_mask.sum()}')

aq_daily = aq_daily[~missing_mask].reset_index(drop=True)
print(f'  AQ daily rows remaining: {len(aq_daily)}')

---
## Parse Weather Data (Open-Meteo) — Aggregate Hourly → Daily

Variables aggregated:
- `temp_mean`       = mean of `temperature_2m` (°C)
- `wind_speed_mean` = mean of `windspeed_10m` (km/h)
- `precip_sum`      = sum of `precipitation` (mm)

The Open-Meteo timestamps are in **local city time** (as specified in the API request  
via the `timezone` parameter). We parse them with timezone-awareness and convert to  
**UTC** for date extraction — consistent with Rule 2.

In [ ]:
def parse_openmeteo(path: Path, city_label: str, tz: str) -> pd.DataFrame:
    with open(path) as f:
        raw = json.load(f)

    hourly = raw['hourly']
    df = pd.DataFrame({
        'datetime':        pd.to_datetime(hourly['time']).tz_localize(tz, ambiguous='infer', nonexistent='shift_forward'),
        'temperature_2m':  hourly['temperature_2m'],
        'windspeed_10m':   hourly['windspeed_10m'],
        'precipitation':   hourly['precipitation'],
    })
    # Convert to UTC and extract date — matches Rule 2
    df['date'] = df['datetime'].dt.tz_convert('UTC').dt.normalize()

    daily = (
        df.groupby('date')
          .agg(
              temp_mean       = ('temperature_2m', 'mean'),
              wind_speed_mean = ('windspeed_10m',  'mean'),
              precip_sum      = ('precipitation',  'sum'),
          )
          .reset_index()
    )
    daily.insert(0, 'city', city_label)
    daily[['temp_mean', 'wind_speed_mean', 'precip_sum']] = \
        daily[['temp_mean', 'wind_speed_mean', 'precip_sum']].round(4)
    return daily


wx_frames = []
for key, cfg in CITIES.items():
    df = parse_openmeteo(RAW_WX / f'weather_{key}_2025Q1.json', cfg['label'], cfg['tz'])
    wx_frames.append(df)
    print(f"{cfg['label']:20s}: {len(df)} daily weather rows")

wx_all = pd.concat(wx_frames, ignore_index=True)
print(f'Total weather rows: {len(wx_all)}')

---
## Merge Air Quality + Weather

In [ ]:
merged = aq_daily.merge(wx_all, on=['city', 'date'], how='inner')
print(f'Merged rows: {len(merged)}')
merged.head(3)

---
## Add Calendar Features

In [ ]:
merged['weekday']    = merged['date'].dt.day_name()
merged['is_weekend'] = merged['date'].dt.dayofweek.isin([5, 6]).astype(int)

COLS = ['city', 'date', 'pm25_mean', 'pm25_p90',
        'temp_mean', 'wind_speed_mean', 'precip_sum',
        'weekday', 'is_weekend']

curated = merged[COLS].sort_values(['city', 'date']).reset_index(drop=True)
print(f'Final curated shape: {curated.shape}')
curated.dtypes

---
## Final Quality Assertions

In [ ]:
# 1. No duplicate (city, date) pairs
dupes = curated.duplicated(subset=['city', 'date']).sum()
assert dupes == 0, f'{dupes} duplicate (city, date) pairs remain!'

# 2. No NaN values in any column
nulls = curated.isnull().sum()
assert nulls.sum() == 0, f'Null values remain:\n{nulls[nulls>0]}'

# 3. pm25_mean > 0 everywhere
assert (curated['pm25_mean'] > 0).all(), 'Non-positive pm25_mean detected!'

print('All quality assertions passed.')
curated.describe().round(2)

---
## Save Curated Dataset

In [ ]:
out_path = CURATED / 'part2_airquality_curated.parquet'
curated.to_parquet(out_path, index=False)
print(f'Saved  : {out_path}')
print(f'Size   : {out_path.stat().st_size / 1024:.1f} KB')
print(f'Rows   : {len(curated)}')
print(f'Columns: {list(curated.columns)}')

---
## QA Summary Report

In [ ]:
print('=' * 55)
print('QA SUMMARY')
print('=' * 55)
print(f'Rule 1 – Exact duplicates removed      : {removed_dupes}')
print(f'Rule 2 – Timezone for aggregation      : UTC')
print(f'Rule 3 – Low-coverage days (< {MIN_OBS} obs)  : {low_coverage_mask.sum()}')
for city in [c['label'] for c in CITIES.values()]:
    n = dropped_per_city.get(city, 0)
    print(f'Rule 4 – Days dropped ({city:13s}): {n}')
print(f'Rule 4 – Total days dropped            : {missing_mask.sum()}')
print('=' * 55)
print(f'Final curated rows                     : {len(curated)}')